# Speech Augmentations

In this seminar, we will learn basic augmentations for speech.

    1) Gaussian Noise
    2) Time Stretching
    3) Pitch Shifting
    4) Volume
    5) Impulse Response
    6) Add Noise on Background
    7) SpecAug (Time/Freq masking, Cutout)

Also pay attention to the `SpeechBrain` library, which contains a lot of useful stuff, including [speech augmentation](https://speechbrain.readthedocs.io/en/latest/API/speechbrain.augment.html)

---

In [ ]:
import librosa
import torch
import torchaudio
import torchaudio.functional as AF
import torchaudio.transforms as T
from IPython import display
from matplotlib import pyplot as plt
from torch import nn

SEED = 42
torch.manual_seed(SEED)

In [ ]:
class AudioVisualizer:
    """Draws waveform and log-mel spectrogram and plays the audio.

    MelSpectrogram transforms are cached by sample rate,
    so they are created once and reused on every call.
    """

    def __init__(
        self,
        figsize: tuple[float, float] = (20, 5),
        n_fft: int = 1024,
        win_length: int = 1024,
        hop_length: int = 256,
        n_mels: int = 80,
    ) -> None:
        self._mel_cache: dict[int, T.MelSpectrogram] = {}
        self.figsize = figsize
        self.mel_kwargs = dict(
            n_fft=n_fft, win_length=win_length, hop_length=hop_length, n_mels=n_mels
        )

    def mel(self, sr: int) -> T.MelSpectrogram:
        if sr not in self._mel_cache:
            self._mel_cache[sr] = T.MelSpectrogram(sample_rate=sr, **self.mel_kwargs)
        return self._mel_cache[sr]

    @torch.no_grad()
    def log_mel(self, wav: torch.Tensor, sr: int) -> torch.Tensor:
        # clamp: log(0) = -inf on silence
        return torch.log(self.mel(sr)(wav).clamp(min=1e-5))

    def show_spec(self, spec: torch.Tensor) -> None:
        plt.figure(figsize=self.figsize)
        plt.imshow(spec.squeeze(), origin="lower", aspect="auto")
        plt.xlabel("Frame", size=20)
        plt.ylabel("Mel bin", size=20)
        plt.show()

    def __call__(self, wav: torch.Tensor, sr: int = 22050, play: bool = True) -> None:
        wav = wav.detach().float().cpu()

        if wav.dim() == 2:
            # Any to mono audio conversion
            wav = wav.mean(dim=0)

        t = torch.arange(wav.shape[-1]) / sr
        plt.figure(figsize=self.figsize)
        plt.plot(t, wav, alpha=0.7, c="green")
        plt.grid()
        plt.xlim(0, wav.shape[-1] / sr)
        plt.xlabel("Time, s", size=20)
        plt.ylabel("Amplitude", size=20)
        plt.show()

        self.show_spec(self.log_mel(wav, sr))

        if play:
            display.display(display.Audio(wav, rate=sr, normalize=False))

In [ ]:
visualize_audio = AudioVisualizer()

In [ ]:
wav, sr = torchaudio.load("LJ001-0001.wav")

# Ground Truth 

In [ ]:
visualize_audio(wav, sr)

# Gaussian Noise

A very simple augmentation that appears in various areas of machine learning.

In [ ]:
generator = torch.Generator().manual_seed(SEED)

noise_std = 0.05

augmented_wav_1 = wav + noise_std * torch.randn(wav.shape, generator=generator)

In [ ]:
visualize_audio(augmented_wav_1, sr)

# Time Stretching

Naive approach:

In [ ]:
# never do this
simple_stretch = wav[:, ::2]
visualize_audio(simple_stretch, sr)

Advanced approach:

This augmentation speeds up or slows down the speed of speech. For more information, visit https://www.dafx.de/paper-archive/2000/pdf/Bernardini.pdf.
There are two ways to do `Time Stretching`:

1) Use `librosa.effects.time_stretch` from `librosa` which works with `wav`s

2) Use `torchaudio.transforms.TimeStretch` from `torchaudio` which works with `spectrogram`s.
In this case, we additionally need to use STFT and iSTFT (inverse STFT)

In [ ]:
stretched = librosa.effects.time_stretch(wav.numpy().squeeze(), rate=2.0)
augmented_wav_2 = torch.from_numpy(stretched)
visualize_audio(augmented_wav_2, sr)

# Pitch Shifting

This augmentation changes the pitch of voice. For example, it makes the voice more "heavy". Similarly, for more information, visit https://www.dafx.de/paper-archive/2000/pdf/Bernardini.pdf.

In [ ]:
for n_steps in (-3, 0, 3):
    shifted = librosa.effects.pitch_shift(wav.numpy().squeeze(), sr=sr, n_steps=n_steps)
    augmented_wav_3 = torch.from_numpy(shifted)
    visualize_audio(augmented_wav_3, sr)

# Volume

This augmentation changes level of volume. It seems like simple augmentation but in some cases it could help :)

In [ ]:
voler = T.Vol(gain=0.1, gain_type="amplitude")
augmented_wav_4 = voler(wav)
visualize_audio(augmented_wav_4, sr)

# Room Impulse Response

This augmentation convolves an audio signal with an impulse response. As a result, we simulate the acoustics of a new environment on the given audio.

It may be a little bit tricky, so feel free to learn more about it:

    1) https://www.acousticalsurfaces.com/blog/acoustics-education/measure-rt60
    2) https://en.wikipedia.org/wiki/Convolution_reverb
    3) https://danielpovey.com/files/2017_icassp_reverberation.pdf

In [ ]:
rir, rir_sr = torchaudio.load("h001_Bedroom_65.wav")
visualize_audio(rir, rir_sr, play=True)

rir = AF.resample(rir, rir_sr, sr)

In [ ]:
def simulate(audio: torch.Tensor, rir: torch.Tensor) -> torch.Tensor:
    rir = rir / rir.norm()
    out = AF.fftconvolve(audio, rir)[..., : audio.shape[-1]]
    return out / out.abs().amax().clamp(min=1.0)

In [ ]:
augmented_wav_5 = simulate(wav, rir)
visualize_audio(augmented_wav_5, sr)

# Add noise audio on background

Let's imagine that we have a lot of samples of noises. For example, sound of street, car, talk and so on. Then we would like to add such noise to the background so that the model (for example, ASR) would be more robust to them.

There is a rule of thumb on how to properly add noise to the background:

    1) https://medium.com/analytics-vidhya/adding-noise-to-audio-clips-5d8cee24ccb8
    2) https://arxiv.org/pdf/1808.00563.pdf (3.1 chapter)

In [ ]:
filename = librosa.ex("trumpet")
noise_np, _ = librosa.load(filename, sr=sr)
noise = torch.from_numpy(noise_np).unsqueeze(0)

visualize_audio(noise, sr)

In [ ]:
def add_background(
    speech: torch.Tensor, noise: torch.Tensor, snr_db: float
) -> torch.Tensor:
    
    if noise.shape[-1] < speech.shape[-1]:
        noise = noise.repeat(1, speech.shape[-1] // noise.shape[-1] + 1)
    start = torch.randint(
        0, noise.shape[-1] - speech.shape[-1] + 1, (1,), generator=generator
    ).item()
    noise = noise[..., start : start + speech.shape[-1]]

    mixed = AF.add_noise(speech, noise, torch.tensor([snr_db]))

    return mixed / mixed.abs().amax().clamp(min=1.0)


snr_db = 20
augmented_wav_6 = add_background(wav, noise, snr_db)

visualize_audio(augmented_wav_6, sr)

# SpecAug (Time/Freq masking, Cutout)

As we already know, main type of “features" in speech processing is (Mel)Spectrogram. From a practical point of view Spectrogram is image, so we could apply some ideas from Computer Vision. `SpecAug` motivation come from it. 

For more details, please, visit: https://arxiv.org/pdf/1904.08779.pdf

In [ ]:
mel_spectrogramer = visualize_audio.mel(sr)

mel_spectrogram = mel_spectrogramer(wav)
log_mel_spectrogram = torch.log(mel_spectrogram.clamp(min=1e-5))
visualize_audio.show_spec(log_mel_spectrogram)

`torchaudio.transforms.FrequencyMasking` add horizontal strip and `torchaudio.transforms.TimeMasking` add vertical strip.

Note that this is one way to implement SpecAug. In practice, you may encounter different implementations.

In [ ]:
specaug = nn.Sequential(
    T.FrequencyMasking(20),
    T.TimeMasking(100),
)

augmented_log_mel_spectrogram = torch.log(specaug(mel_spectrogram).clamp(min=1e-5))

visualize_audio.show_spec(augmented_log_mel_spectrogram)